# DEEP LEARNING – ĐỀ A

**Thời gian làm bài:** 50 phút  
**Môi trường thực hành:** Google Colab  
**Hình thức:** Thực hành lập trình trên notebook `.ipynb`

## Thông tin sinh viên

**MSSV:**  22301763
**Họ tên:** Tăng Tiến Phát

## Yêu cầu chung

1. Làm bài trực tiếp trên Google Colab.
2. Không xóa các ô Markdown yêu cầu.
3. Viết code trong các ô Code ngay sau từng ô Markdown.
4. File nộp phải là notebook `.ipynb` đã chạy với đầy đủ output.


In [16]:
# Cài đặt và import thư viện cần thiết
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [17]:
# Các hàm hỗ trợ dùng chung

def vectorize_sequences(sequences, dimension):
    """Chuyển list các sequence số nguyên thành ma trận multi-hot."""
    results = np.zeros((len(sequences), dimension), dtype="float32")
    for i, sequence in enumerate(sequences):
        valid_indices = [index for index in sequence if index < dimension]
        results[i, valid_indices] = 1.0
    return results


def plot_training_validation_loss(history, title):
    """Vẽ training loss và validation loss."""
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]
    epochs = range(1, len(loss) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, loss, "bo-", label="Training loss")
    plt.plot(epochs, val_loss, "ro-", label="Validation loss")
    plt.title(title)
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_training_validation_accuracy(history, title):
    """Vẽ training accuracy và validation accuracy."""
    acc = history.history["accuracy"]
    val_acc = history.history["val_accuracy"]
    epochs = range(1, len(acc) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, acc, "bo-", label="Training accuracy")
    plt.plot(epochs, val_acc, "ro-", label="Validation accuracy")
    plt.title(title)
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()


def get_best_epoch_by_val_loss(history):
    """Chọn epoch có validation loss thấp nhất."""
    val_loss = history.history["val_loss"]
    best_epoch = int(np.argmin(val_loss) + 1)
    best_val_loss = float(np.min(val_loss))
    return best_epoch, best_val_loss


def plot_confusion_matrix_result(y_true, y_pred, class_names, title):
    """Vẽ confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(7, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap="Blues", values_format="d", ax=ax)
    plt.title(title)
    plt.xticks(rotation=45)
    plt.show()

# Phần 1. Phân loại nhị phân với IMDB

## 1.1. Load dữ liệu

Load bộ dữ liệu IMDB bằng Keras với:

- `num_words = 10000`
- Dữ liệu dùng để phân loại review phim thành hai lớp: negative và positive.
- In số lượng mẫu train, test và xem thử một review đã được mã hóa.

In [1]:
from tensorflow.keras.datasets import imdb

NUM_WORDS_IMDB = 10000

# TODO: Load dữ liệu IMDB với num_words=NUM_WORDS_IMDB
# (train_data, train_labels), (test_data, test_labels) = ...
(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=10000)
# TODO: In số lượng mẫu train/test
print(len(train_data), len(test_data))
# TODO: In thử 20 token đầu tiên của một review
print(train_data[0][:20])
# TODO: In nhãn tương ứng
print(train_labels[0])

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
25000 25000
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25]
1


## 1.2. Chuẩn hóa dữ liệu IMDB

Chuẩn hóa dữ liệu review sang dạng vector multi-hot:

- Mỗi review trở thành vector kích thước `10000`.
- Chuyển nhãn sang kiểu `float32`.
- In shape của `x_train`, `x_test`, `y_train`, `y_test`.

In [18]:
# TODO: Chuẩn hóa train_data và test_data bằng hàm vectorize_sequences
# x_train = ...
# x_test = ...
def vectorize_sequences(sequences, dimension=10000):
x_train = vectorize_sequences(train_data, 10000)
x_test = vectorize_sequences(test_data, 10000)

# TODO: Chuyển labels sang float32
# y_train = ...
# y_test = ...
y_train = np.asarray(train_labels).astype('float32')
y_test = np.asarray(test_labels).astype('float32')
# TODO: In shape của các biến
print(x_train.shape)
print(x_test.shape)

IndentationError: expected an indented block after function definition on line 4 (2455215475.py, line 5)

## 1.3. Tách validation set cho IMDB

Tách dữ liệu validation từ training set:

- Lấy `10000` mẫu đầu tiên làm validation set.
- Phần còn lại dùng để train thăm dò.
- In shape của các tập sau khi tách.

In [19]:
# TODO: Tách validation set
# x_val = x_train[:10000]
x_val = x_train[:10000]
# partial_x_train = x_train[10000:]
partial_x_train = x_train[10000:]
# y_val = y_train[:10000]
y_val = y_train[:10000]
# partial_y_train = y_train[10000:]
partial_y_train = y_train[10000:]
# TODO: In shape sau khi tách
print(x_val.shape)
print(partial_x_train.shape)
print(y_val.shape)
print(partial_y_train.shape)

NameError: name 'x_train' is not defined

## 1.4. Tạo model phân loại nhị phân

Tạo model Keras theo cấu trúc:

- Lớp ẩn 1: có 16 nơ-ron, hàm kích hoạt là relu  
- Lớp ẩn 2: có 16 nơ-ron, hàm kích hoạt là relu  
- Lớp ouput: cấu hình thích hợp

Compile model với:

- optimizer: `rmsprop`
- loss: `binary_crossentropy`
- metric: `accuracy`

Hiển thị `model.summary()`.

In [20]:
def build_imdb_model():
    # TODO: Tạo Sequential model theo yêu cầu
    # model = keras.Sequential([...])
model = keras.Sequential([
    layers.Dense(16, activation='relu', input_shape=(10000,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
    # TODO: Compile model
    # model.compile(...)
model.compile(optimizer='rmsprop',
              loss='binary_crossentropy',
              metrics=['accuracy'])
    # return model
    pass
# TODO: Tạo model và hiển thị summary
# imdb_model = build_imdb_model()

# imdb_model.summary()
model.summary()

IndentationError: expected an indented block after function definition on line 1 (2201409355.py, line 4)

## 1.5. Train thăm dò model IMDB

Train thăm dò model với:

- `epochs = 20`
- `batch_size = 512`
- Có sử dụng validation set đã tách.

Lưu kết quả huấn luyện vào biến `history_imdb`.

In [21]:
EPOCHS_EXPLORATION_IMDB = 20
BATCH_SIZE_IMDB = 512

# TODO: Train thăm dò model
# history_imdb = imdb_model.fit(
#     partial_x_train,
#     partial_y_train,
#     epochs=EPOCHS_EXPLORATION_IMDB,
#     batch_size=BATCH_SIZE_IMDB,
#     validation_data=(x_val, y_val)
# )
history = model.fit(partial_x_train,
                    partial_y_train,
                    epochs=20,
                    batch_size=512,
                    validation_data=(x_val, y_val))

NameError: name 'model' is not defined

## 1.6. Vẽ training loss và validation loss

Vẽ biểu đồ gồm:

- Training loss
- Validation loss

Dựa vào biểu đồ, nhận xét thời điểm model bắt đầu overfitting.

In [15]:
# TODO: Vẽ training loss và validation loss
# plot_training_validation_loss(history_imdb, "IMDB - Training Loss và Validation Loss")
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(1, len(loss) + 1)

plt.plot(epochs, loss, 'bo-', label='Training loss')
plt.plot(epochs, val_loss, 'r^-', label='Validation loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

NameError: name 'history' is not defined

## 1.7. Xác định số epoch phù hợp

Dựa vào biểu đồ validation loss:

- Chọn số epoch phù hợp.
- Gán vào biến `BEST_EPOCHS_IMDB`.
- In giá trị đã chọn.

In [22]:
# TODO: Quan sát biểu đồ và gán số epoch phù hợp
BEST_EPOCHS_IMDB = None
def get_best_epoch_by_val_loss(history):
    """Chọn epoch có validation loss thấp nhất."""
    val_loss = history.history["val_loss"]
    best_epoch = int(np.argmin(val_loss) + 1)
    best_val_loss = float(np.min(val_loss))
    return best_epoch
BEST_EPOCHS_IMDB = get_best_epoch_by_val_loss(history)
print("Số epoch phù hợp cho IMDB là:", BEST_EPOCHS_IMDB)

NameError: name 'history' is not defined

## 1.8. Train lại model IMDB

Tạo lại model mới từ đầu và train trên toàn bộ training set `x_train`, `y_train` với số epoch đã chọn.

Lưu model cuối cùng vào biến `final_imdb_model`.

In [23]:
# TODO: Tạo lại model mới
# final_imdb_model = build_imdb_model()
final_imdb_model = build_imdb_model()
# TODO: Train lại trên toàn bộ training set
final_imdb_model.fit(
   x_train,
     y_train,
    epochs=BEST_EPOCHS_IMDB,
   batch_size=BATCH_SIZE_IMDB
)

NameError: name 'build_imdb_model' is not defined

## 1.9. Vẽ confusion matrix cho IMDB

Dùng model cuối cùng để dự đoán trên test set:

- Dùng `predict()`.
- Chuyển xác suất thành nhãn 0/1 với ngưỡng 0.5.
- Vẽ confusion matrix với hai lớp: `Negative`, `Positive`.

In [24]:
# TODO: Dự đoán xác suất trên test set
# imdb_probabilities = final_imdb_model.predict(x_test).ravel()
imdb_probabilities = final_imdb_model.predict(x_test).ravel()

# TODO: Chuyển xác suất thành nhãn
# imdb_predictions = ...
indb_predictions = (imdb_probabilities >= 0.5).astype(int)
# TODO: Vẽ confusion matrix
# plot_confusion_matrix_result(
plot_confusion_matrix_result(
    y_true=y_test,
    y_pred=indb_predictions,
    class_names=["Negative", "Positive"],
    title="Confusion Matrix - IMDB"
)

NameError: name 'final_imdb_model' is not defined

## 1.10. Test model IMDB

Đánh giá model trên test set:

- In test loss.
- In test accuracy.
- In `classification_report`.

In [ ]:
# TODO: Evaluate model trên test set
 test_loss_imdb, test_acc_imdb = final_imdb_model.evaluate(x_test, y_test)

# TODO: In kết quả
print("IMDB test loss:", test_loss_imdb)
print("IMDB test accuracy:", test_acc_imdb)

# TODO: In classification report

 print(classification_report(...))

# Phần 2. Phân loại 5 lớp với Reuters

## 2.1. Load dữ liệu Reuters

Load bộ dữ liệu Reuters với:

- `num_words = 10000`
- Chỉ sử dụng 5 lớp: `[2, 4, 6, 8, 10]`

In số lượng mẫu train, test ban đầu và số lớp ban đầu.

In [25]:
from tensorflow.keras.datasets import reuters

NUM_WORDS_REUTERS = 10000
SELECTED_CLASSES = [2, 4, 6, 8, 10]

# TODO: Load dữ liệu Reuters
 (train_data_reuters, train_labels_reuters), (test_data_reuters, test_labels_reuters) = ...

# TODO: In số lượng mẫu train/test ban đầu
print("Số lượng mẫu train/test ban đầu:", len(train_data_reuters), len(test_data_reuters))
# TODO: In số lớp ban đầu
print("Số lớp ban đầu:", len(np.unique(train_labels_reuters)))

IndentationError: unexpected indent (4156352490.py, line 7)

## 2.2. Lọc Reuters còn 5 lớp

Lọc dữ liệu Reuters để chỉ giữ các mẫu thuộc các lớp:

`[2, 4, 6, 8, 10]`

Sau khi lọc:

- Ánh xạ nhãn cũ về nhãn mới `0, 1, 2, 3, 4`.
- In số lượng mẫu train/test sau khi lọc.
- In phân bố nhãn sau khi lọc.

In [ ]:
# Không sửa hàm này
def filter_reuters_classes(data, labels, selected_classes):
    """Lọc dữ liệu chỉ giữ các mẫu thuộc các lớp trong selected_classes"""
    # Chuyển labels sang numpy array
    labels = np.asarray(labels)
    data = np.asarray(data, dtype=object)
    # Tạo mask để giữ các label thuộc selected_classes
    mask = np.isin(labels, selected_classes)
    # Lọc data và labels
    filtered_data = data[mask]
    filtered_labels = labels[mask]
    # Ánh xạ nhãn cũ sang nhãn mới 0..4
    class_to_new_label = {old_label: new_label for new_label, old_label in enumerate(selected_classes)}
    mapped_labels = np.array([class_to_new_label[label] for label in filtered_labels], dtype="int64")
    # Return filtered_data, mapped_labels
    return filtered_data.tolist(), mapped_labels

# TODO: Lọc train và test
# train_data_5, train_labels_5 = ...
train_data_5, train_labels_5 = filter_reuters_classes(train_data_reuters, train_labels_reuters, SELECTED_CLASSES)
# test_data_5, test_labels_5 = ...
test_data_5, test_labels_5 = filter_reuters_classes(test_data_reuters, test_labels_reuters, SELECTED_CLASSES)
# TODO: In số lượng mẫu và phân bố nhãn
print("Số lượng mẫu train/test sau khi lọc:", len(train_data_5), len(test_data_5))
print("Phân bố nhãn sau khi lọc:")
print(np.unique(train_labels_5, return_counts=True))

## 2.3. Chuẩn hóa dữ liệu Reuters

Chuẩn hóa dữ liệu Reuters:

- Vector hóa mỗi bản tin thành vector multi-hot kích thước `10000`.
- Chuyển labels sang kiểu `int64`.
- In shape của dữ liệu sau khi chuẩn hóa.

In [ ]:
# TODO: Vector hóa dữ liệu Reuters
# x_train_5 = ...
# x_test_5 = ...

# TODO: Chuyển labels sang numpy array kiểu int64
# y_train_5 = ...
# y_test_5 = ...

# TODO: In shape

## 2.4. Tách validation set cho Reuters

Tách validation set từ training set Reuters:

- Trộn dữ liệu trước khi tách.
- Dùng `20%` dữ liệu train làm validation set.
- Phần còn lại dùng để train thăm dò.

In [ ]:
# TODO: Trộn dữ liệu bằng np.random.permutation hoặc np.random.default_rng
# TODO: Tách 20% dữ liệu train làm validation

VALIDATION_RATIO_REUTERS = 0.2

# x_val_5 = ...
# y_val_5 = ...
# partial_x_train_5 = ...
# partial_y_train_5 = ...

# TODO: In shape sau khi tách

## 2.5. Tạo model phân loại 5 lớp

Tạo model Keras theo cấu trúc:

- Lớp ẩn 1: có 64 nơ-ron, hàm kích hoạt là relu  
- Lớp ẩn 2: có 64 nơ-ron, hàm kích hoạt là relu  
- Lớp ouput: cấu hình thích hợp

Compile model với:

- optimizer: `rmsprop`
- loss: `sparse_categorical_crossentropy`
- metric: `accuracy`

Hiển thị `model.summary()`.

In [ ]:
def build_reuters_5class_model():
    # TODO: Tạo Sequential model theo yêu cầu
    # model = keras.Sequential([...])

    # TODO: Compile model
    # model.compile(...)

    # return model
    pass


# TODO: Tạo model và hiển thị summary
# reuters_model = build_reuters_5class_model()
# reuters_model.summary()

## 2.6. Train thăm dò model Reuters 5 lớp

Train thăm dò model với:

- `epochs = 20`
- `batch_size = 256`
- Có sử dụng validation set đã tách.

Lưu kết quả huấn luyện vào biến `history_reuters`.

In [ ]:
EPOCHS_EXPLORATION_REUTERS = 20
BATCH_SIZE_REUTERS = 256

# TODO: Train thăm dò model Reuters 5 lớp
# history_reuters = reuters_model.fit(
#     partial_x_train_5,
#     partial_y_train_5,
#     epochs=EPOCHS_EXPLORATION_REUTERS,
#     batch_size=BATCH_SIZE_REUTERS,
#     validation_data=(x_val_5, y_val_5)
# )

## 2.7. Vẽ training loss và validation loss cho Reuters

Vẽ biểu đồ gồm:

- Training loss
- Validation loss

Dựa vào biểu đồ, nhận xét thời điểm model bắt đầu overfitting.

In [ ]:
# TODO: Vẽ training loss và validation loss
# plot_training_validation_loss(history_reuters, "Reuters 5 lớp - Training Loss và Validation Loss")

## 2.8. Xác định số epoch phù hợp cho Reuters

Dựa vào biểu đồ validation loss:

- Chọn số epoch phù hợp.
- Gán vào biến `BEST_EPOCHS_REUTERS`.
- In giá trị đã chọn.

In [ ]:
# TODO: Quan sát biểu đồ và gán số epoch phù hợp
BEST_EPOCHS_REUTERS = None

print("Số epoch phù hợp cho Reuters 5 lớp là:", BEST_EPOCHS_REUTERS)

## 2.9. Train lại model Reuters 5 lớp

Tạo lại model mới từ đầu và train trên toàn bộ training set đã lọc với số epoch đã chọn.

Lưu model cuối cùng vào biến `final_reuters_model`.

In [ ]:
# TODO: Tạo lại model mới
# final_reuters_model = build_reuters_5class_model()

# TODO: Train lại trên toàn bộ dữ liệu train đã lọc
# final_reuters_model.fit(
#     x_train_5,
#     y_train_5,
#     epochs=BEST_EPOCHS_REUTERS,
#     batch_size=BATCH_SIZE_REUTERS
# )

## 2.10. Confusion matrix và test model Reuters

Đánh giá model Reuters trên test set:

1. Dùng `predict()` để dự đoán.
2. Dùng `np.argmax()` để lấy nhãn dự đoán.
3. Vẽ confusion matrix.
4. In test loss, test accuracy.
5. In `classification_report`.

In [ ]:
# TODO: Dự đoán trên test set
# reuters_probabilities = final_reuters_model.predict(x_test_5)
# reuters_predictions = ...

# TODO: Vẽ confusion matrix
# class_names = [f"Class {i}" for i in range(5)]
# plot_confusion_matrix_result(
#     y_true=y_test_5,
#     y_pred=reuters_predictions,
#     class_names=class_names,
#     title="Confusion Matrix - Reuters 5 lớp"
# )

# TODO: Evaluate model trên test set
# test_loss_reuters, test_acc_reuters = final_reuters_model.evaluate(x_test_5, y_test_5)

# TODO: In kết quả
# print("Reuters test loss:", test_loss_reuters)
# print("Reuters test accuracy:", test_acc_reuters)

# TODO: In classification report
# print(classification_report(...))